# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv openai

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Prepare the dataset

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [3]:
dataset_id = config.get_config_value("LIGHTNINGROD_DATASET_ID")

dataset = lr.datasets.get(dataset_id)
_ = dataset.download()


In [4]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(90, None)),
    split=SplitParams(test_size=0.2),
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 435 samples                                                                                    │
│                                                                                                                 │
│    Filter:  Dropped 53 invalid, 245 horizon → 137 remain                                                        │
│    Dedup:   137 remain (0 duplicates)                                                                           │
│    Split:   Splits: 34 train | 28 test (0 dropped, no prediction_date)                                          │
│             75 train samples removed for leakage                                                                │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  75/109 train samples (68%) were removed for temporal leakage — the date_close or resolution_date of train      │
│  questions extends into the test period.                                                                        │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Extend the seed generator (date) filter range to start earlier — questions generated near the start      │
│  will resolve well before the test window. Aim for at least 2× your max resolution horizon.                     │
│      • Generate more samples by increasing max_seeds in lr.transforms.run() or removing the limit, or increase  │
│  questions_per_seed in your question generator config. A larger, temporally well-spread dataset naturally       │
│  pushes the split cutoff far enough back.                                                                       │
│      • If very few seeds were returned by the pipeline (check the run summary table), the search queries may    │
│  not surface results across the full date range. Try more diverse search queries, increase                      │
│  articles_per_search, or shorten interval_duration_days.                                                        │
│                                                                                                                 │
│  Only 34 train samples remain after preparation. This is below the recommended minimum of +1000 for effective   │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_seeds in lr.transforms.run() to generate more samples.                                      │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [5]:
from lightningrod import GRPOTrainingConfig

config = GRPOTrainingConfig(
    base_model_id="openai/gpt-oss-120b",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.11
Effective steps: 2
Train tokens: 151,348
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.


In [6]:
training_job = lr.training.run(config, dataset=train_dataset, name="Forecasting fine-tune")
print(f"Job {training_job.id} completed with status: {training_job.status}")
print(f"Trained model ID: {training_job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job ID: 3c25ab6d-6ce3-4326-a4db-4c4e23c1d244                                                                 │
│                                                                                                                 │
│    Model:                                                                                                       │
│  checkpoint:MmI5ODFhNDgtOTgzZS01NGMyLTkzMmQtZjllOGQzNTEzMDg4OnRyYWluOjAvc2FtcGxlcl93ZWlnaHRzL3N0ZXBfMDAwMg      │
│                                                                                                                 │
│    reward: latest -0.0190  avg -0.0747  (2 steps)  (higher is better)                                           │
│        ▁█                                                                                                       │
│    mean_output_tokens: latest 437.2500  avg 478.8672  (2 steps)                                                 │
│        █▁                                                                                                       │
│                                                                                                                 │
│    Cost:  $0.17                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 3c25ab6d-6ce3-4326-a4db-4c4e23c1d244 completed with status: COMPLETED
Trained model ID: checkpoint:MmI5ODFhNDgtOTgzZS01NGMyLTkzMmQtZjllOGQzNTEzMDg4OnRyYWluOjAvc2FtcGxlcl93ZWlnaHRzL3N0ZXBfMDAwMg


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model. You can also use the OpenAI-compatible API directly — see [08_foresight_model.ipynb](08_foresight_model.ipynb) for the pre-trained foresight model.


In [7]:
print(lr.predict(training_job.model_id, "Will the Fed cut rates by 25bp in March 2026?"))


**Short answer:**  
At this point (early May 2024) it is **impossible to say with certainty** whether the Federal Reserve will deliver a 25‑basis‑point (bp) rate cut in the March 2026 FOMC meeting. The best we can do is outline the economic forces that will shape the decision, look at the current market pricing of that outcome, and give a rough probability range based on the available data and historical patterns.

Below is a structured “forecast‑framework” you can use to keep track of the variables that will matter over the next two years, plus a snapshot of the **current market‑based probability** of a March 2026 cut.

---

## 1.  Where the Fed stands today (May 2024)

| Indicator | Latest reading (April 2024) | Fed’s stated stance |
|-----------|----------------------------|----------------------|
| **Policy rate** | 5.25‑5.50 % (target range) | “Restrictive but appropriate” |
| **Core PCE inflation (YoY)** | 4.0 % | Above the 2 % goal – still too high |
| **Headline PCE inflation**

## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset, reports metrics, and can include a reasoning comparison between the base and fine-tuned model. Use the same dataset for a quick check, or a separate test split for production.

In [8]:
from lightningrod import training

eval_job = lr.evals.run_from_training_job(
    config,
    training_job,
    test_dataset,
    reasoning_comparison_sample_size=20,
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Job ID: 8ce09d74-28a6-48f1-82e0-ed8833008b55                                                                 │
│    Dataset: ad98afcb-da82-4f7b-ada5-5447122e992c                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┓                                                                 │
│  ┃ Metric              ┃    Base ┃ Fine-tuned ┃                                                                 │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━┩                                                                 │
│  │ brier_score         │  0.1850 │     0.1899 │                                                                 │
│  │ ece                 │  0.2543 │     0.1186 │                                                                 │
│  │ mc_ece              │       — │          — │                                                                 │
│  │ mean_reward         │ -0.1850 │    -0.1899 │                                                                 │
│  │ mean_valid_reward   │ -0.1850 │    -0.1899 │                                                                 │
│  │ n_samples           │      28 │         28 │                                                                 │
│  │ n_valid             │      28 │         28 │                                                                 │
│  │ parse_rate          │  1.0000 │     1.0000 │                                                                 │
│  │ total_cost          │  0.0121 │     0.0110 │                                                                 │
│  │ total_input_tokens  │   28924 │      28924 │                                                                 │
│  │ total_output_tokens │   15596 │      13124 │                                                                 │
│  └─────────────────────┴─────────┴────────────┘                                                                 │
│                                                                                                                 │
│  Reasoning comparison analysis                                                                                  │
│  1) Overall verdict (2-4 sentences)                                                                             │
│  The trained model demonstrates overall improved calibration, slightly more structured reasoning, and a         │
│  greater use of quantified uncertainties across problem types, especially where base model responses were less  │
│  precise or more speculative. However, in most cases, the improvements are incremental rather than              │
│  transformative: both models exhibit similar strengths in factual grounding and internal consistency, but       │
│  still show gaps in explicit assumption articulation and actionable takeaway clarity. Across diverse scenarios  │
│  (military events, policy actions), training led to modest gains in calibration language, but persistent        │
│  reasoning limitations remain. The trained model does not consistently outperform: in a handful of cases,       │
│  probability estimates appear more conservative, but not necessarily more accurate, with little difference in   │
│  practical actionability of outputs.                                                                            │
│                                                       

In [ ]:
lr.evals.download_results(eval_job.id, "./eval-results")

> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.